# **Introduction**

This is the python code for the project Smart Soil: IoT-Enabled Parametric Agriculture Insurance on Blockchain. The goal of this code is to successfully interpret the data received through moisture sensor and trigger partial or full insurance payments based on the stage of the moisture reached. Each stage and the subsequent payment of that stage is mentioned in the main body of the code. We used Ganache for our Ethereum network and Remix IDE for solidity contract deployment. The code is explained and well commented using "#" where necesary.

# **Table of Contents**

1. Library imports
2. Establishing Blockchain Connection
3. Hardware Calibration
4. Payout function
5. Main Body of the Code
6. References

# **1. Library imports**

In [ ]:
# Organizing all the required imports
import board
import time
import select
import busio
import sys
from web3 import Web3
from adafruit_ads1x15.analog_in import AnalogIn
import adafruit_ads1x15.ads1115 as ADFS

# **2. Establishing Blockchain Connection**

References for the subsequent code: [[1]](#1) , [[2]](#2) ,[[3]](#3)

In [ ]:
# Defining currency address and initializing connection to blockchain
URL_GANACHE = "http://192.168.0.18:7545"
w3 = Web3(Web3.HTTPProvider(URL_GANACHE)) # CHANGE: Added our own URL

# Providing all required addresses
PRIVATE_KEY_INSURANCE = "0x7af60d91b9c596e3eeca8d4198fe00ec6008267615e4bfde8236acd1184ca9b2"
ADDRESS_INSURANCE = "0xeA41EeA8dcf9518bC5e5aeA5Aebb47a17867c703"
ADDRESS_CONTRACT= "0x7451a7544A280AfdD11d089Cb5b1843D195C5aD6"

# Setting up an Application Binary Interface for Python and Solidity communication
ABI_LIST = [{"inputs": [{"internalType": "uint8", "name": "riskType", "type": "uint8"}, {"internalType": "uint256", "name": "payoutAmountWei", "type": "uint256"}, {"internalType": "string", "name": "stageName", "type": "string"}],"name": "reportRisk","outputs": [],"stateMutability": "nonpayable","type": "function"}] # CHANGE: Name and parameters

# Creating an object to define the deployed smart contract
contract = w3.eth.contract(address = ADDRESS_CONTRACT, abi = ABI_LIST) # CHANGE: Variable names

# **3. Hardware Calibration**

References for the subsequent code: [[4]](#4) , [[5]](#5) , [[6]](#6) , [[7]](#7) , [[8]](#8)

In [ ]:
# Setting up the hardware

# Communication protocol for ADS1115 and analog to digital data conversion
# CHANGE: Variable names, added board reference to SCL and SDA, inserted ADFS since we did not import ADS1115 directly
i2c_bus_object = busio.I2C(board.SCL, board.SDA)
ads = ADFS.ADS1115(i2c_bus_object)
sensor_analog_input= AnalogIn(ads, 0)
# ENDCHANGE

# Calibrating the sensor for absolute dry or wet values
WET_VALUE = 11000
DRY_VALUE = 26000
DURATION_COUNT =5

#Converting raw sensor values into a percentage amount
def moisture_value_percentage():
   raw_data = sensor_analog_input.value
   percentage_value = ((raw_data - DRY_VALUE) / (WET_VALUE - DRY_VALUE)) * 100
   return max(0, min(100, percentage_value))

# A protocol to stop the terminal without crashing
def if_key_press(): #CHANGE: function name
   return bool(select.select([sys.stdin], [], [], 0)[0]) if sys.platform != "win32" else False

# **4. Payout function**

References for the subsequent code: [[9]](#9) , [[10]](#10) ,[[11]](#11) , [[12]](#12) ,[[13]](#13) , [[14]](#14)

In [ ]:
# Defining the payout function

# Defining the function to create, sign, and send blockchain transactions
def send_payout(payout_type, eth_amount, stage_level, current_val):
   print(f"\n{payout_type} {stage_level} DETECTED")
   try:
       # nonce makes sure there are no connection errors on rapid transactions
       nonce = w3.eth.get_transaction_count(ADDRESS_INSURANCE, "pending") # CHANGE: added "pending" to count even the transactions that are not yet mined
       risk_code = 0 if payout_type == "DROUGHT" else 1
       wei_amount = w3.to_wei(eth_amount, "ether") # CHANGE: stored the converted value in a variable

       # Accessing the deployed smart contract and calling the reportRisk function with arguments:
       # CHANGE: Variable names, chainID
       transaction = contract.functions.reportRisk(risk_code, wei_amount, stage_level).build_transaction({
           "chainId": 1337,
           "gas": 300000,
           "gasPrice": w3.eth.gas_price,
           "from": ADDRESS_INSURANCE,
           "nonce": nonce,
       })
       # ENDCHANGE

       # Signing the transaction using the private key to prove that the transaction was approved by the owner
       # CHANGE: Variable names, added print statement to display hash generated, added exception in case an exception occurs in try and blockchain fails
       signed_transaction = w3.eth.account.sign_transaction(transaction, private_key=PRIVATE_KEY_INSURANCE)
       transaction_hash = w3.eth.send_raw_transaction(signed_transaction.raw_transaction)
       print(f"SUCCESSFUL: {stage_level} payment sent. Hash: {w3.to_hex(transaction_hash)}")
       return True
   except Exception as e:
       print(f"Blockchain Error: {e}")
       return False
       #ENDCHANGE

# **5. Main Body of the Code**

References for the subsequent code: [[11]](#11) , [[15]](#15) , [[7]](#7) , [[16]](#16) , [[17]](#17)

In [1]:
# Main body of the code, it describes the basic structure of the smart contract conditions

# Describing Initial values of counter, zone, and payment
consecutive_count = 0
last_zone = "INITIAL"
last_payment_stage = 0

print("Processing.. Press any key to stop.")

try:
   while True:
       # Function to stop the program with a key press and terminate without crashing
       if if_key_press(): break

       moisture_level = moisture_value_percentage()

       # Assigning a unique alphanumeric code to each stage. D1 to D3 are drought stages. F1 to F3 are flood stages. 3rd stage is final and total disaster in each case.
       if moisture_level < 10.0:
           moisture_zone = "D3"
       elif 10.0 <= moisture_level < 20.0:
           moisture_zone = "D2"
       elif 20.0 <= moisture_level < 30.0:
           moisture_zone = "D1"
       elif 30.0 <= moisture_level <= 80.0:
           moisture_zone = "OKAY"
       elif 80.0 < moisture_level <= 85.0:
           moisture_zone = "F1"
       elif 85.0 < moisture_level <= 90.0:
           moisture_zone = "F2"
       else:
           moisture_zone = "F3"

       # Making sure that the counter will increment only if we stay in the same stage else it will reset
       if moisture_zone == last_zone:
           consecutive_count += 1
       else:
           consecutive_count = 1
           last_zone = moisture_zone

       # Defining moisture level statuses
       status_label ="OKAY"
       if "D" in moisture_zone: status_label = "DRY"
       if "F" in moisture_zone: status_label = "WET"

       print(f"Moisture Status: {status_label} ({moisture_level:.1f}%) Count: {consecutive_count}")

       # Making sure to proceed with payout logic only if in the danger zone for five consecutive counts and we are not in the "OKAY" zone.
       if consecutive_count>= DURATION_COUNT and moisture_zone != "OKAY":

           # Getting the stage number from alphanumerical stage code
           stage_number = int(moisture_zone[1])
           payout_type = "DROUGHT" if "D" in moisture_zone else "FLOOD"

           # Defining the actual payment amount Details
           if stage_number == 3:
               eth, transfer_amount =2, "Stage 3 (FULL PAYMENT)"
           elif stage_number == 2:
               eth, transfer_amount = 1, "Stage 2 (PARTIAL PAYMENT)"
           else:
               eth, transfer_amount = 1, "Stage 1 (PARTIAL PAYMENT)"

           #Execute payout only if the stage is actually higher than the last stage payment we made
           if stage_number > last_payment_stage:
               if send_payout(payout_type, eth, transfer_amount, moisture_level):
                   last_payment_stage = stage_number

                   # If Full Payment condition is met, terminate the program
                   if stage_number == 3:
                       print("Complete disaster. Full Payment transferred. Program terminating..")
                       sys.exit()

                   consecutive_count=0
                   
       # This represents and ensures the counter measure seconds            
       time.sleep(1)

# A force stop button to stop the loop
except KeyboardInterrupt:
   print("\nProgram stopped.")

Processing... Press any key to stop.


NameError: name 'if_key_press' is not defined

# **6. References**
<a id="1">[1]</a> 
Ethereum Foundation. (n.d.). Overview web3.py 7.14.1 documentation. Read the Docs. Retrieved from https://web3py.readthedocs.io/en/latest/overview.html

<a id="2">[2]</a> 
Kumar, S. (2023, March 16). ABI in Solidity. DEV Community. https://dev.to/shlok2740/abi-in-solidity-11a7

<a id="3">[3]</a> 
Web3.js. (n.d.). ABI. https://docs.web3js.org/libdocs/ABI/

<a id="4">[4]</a> 
Adafruit Industries. (n.d.). Adafruit CircuitPython ADS1x15 library. https://docs.circuitpython.org/projects/ads1x15/en/stable/

<a id="5">[5]</a> 
Adafruit Industries. (n.d.). busio Hardware accelerated external bus access. https://docs.circuitpython.org/en/latest/shared-bindings/busio/

<a id="6">[6]</a> 
Clark, L. (2025, October 13). Python & CircuitPython | Adafruit 4 - channel ADC breakouts. Adafruit Learning System. https://learn.adafruit.com/adafruit-4-channel-adc-breakouts/python-circuitpython

<a id="7">[7]</a> 
Python Software Foundation. (n.d.). sys-System-specific parameters and functions. https://docs.python.org/3/library/sys.html

<a id="8">[8]</a> 
Python Software Foundation. (n.d.). select-Waiting for I/O completion. https://docs.python.org/3/library/select.html

<a id="9">[9]</a>
Ethereum Foundation. (n.d.). Web3 API: Currency conversions. Web3.py. https://web3py.readthedocs.io/en/stable/web3.main.html#web3.Web3.to_wei

<a id="10">[10]</a>
Ethereum Foundation. (n.d.). Eth API: Transactions. Web3.py. https://web3py.readthedocs.io/en/stable/web3.eth.html#web3.eth.Eth.get_transaction_count

<a id="11">[11]</a>
Python Software Foundation. (n.d.). Compound statements: The try statement. https://docs.python.org/3/reference/compound_stmts.html#the-try-statement

<a id="12">[12]</a>
Ethereum Foundation. (n.d.). Eth Account: Sign a transaction. Web3.py. https://web3py.readthedocs.io/en/stable/web3.eth.account.html#web3.eth.account.Account.sign_transaction

<a id="13">[13]</a>
Ethereum Foundation. (n.d.). Web3 API: Hex conversions. Web3.py. https://web3py.readthedocs.io/en/stable/web3.main.html#web3.Web3.to_hex

<a id="14">[14]</a>
Ethereum Foundation. (n.d.). Eth API: Transactions. Web3.py. https://web3py.readthedocs.io/en/stable/web3.eth.html#web3.eth.Eth.send_raw_transaction

<a id="15">[15]</a>
Python Software Foundation. (n.d.). More control flow tools. https://docs.python.org/3/tutorial/controlflow.html

<a id="16">[16]</a>
Python Software Foundation. (n.d.). time-Time access and conversions: time.sleep. https://docs.python.org/3/library/time.html#time.sleep

<a id="17">[17]</a>
Python Software Foundation. (n.d.). Built - in exceptions: KeyboardInterrupt. https://docs.python.org/3/library/exceptions.html#KeyboardInterrupt